In [1]:
reports = [
"""
Patient: DUPONT Jean
IPP: 123456
Examen: Hémoglobine
Résultat: 13.2 g/dL
Valeurs de référence: 12.0 - 16.0 g/dL
Statut: Normal
""",

"""
IPP: 789101
Patient: MARTIN Claire
Examen: CRP
Résultat: 18 mg/L
Valeurs de référence: < 5 mg/L
Conclusion: Anormal
""",

"""
Patient: Durand Paul
IPP: 456999
Examen: Glycémie
Résultat: 1.32 g/L
Valeurs de référence: 0.70 - 1.10 g/L
"""
]

In [26]:
import re

patient_pattern = ["patient"]
ipp_pattern = ["ipp"]
examen_pattern = ["examen"]
result_pattern = ["résultat"]
val_ref_pattern = ["référence"]
status_pattern = ["statut", "conclusion"]


def clean_report(text: str) -> str:
    return text.lower().strip()

def build_report(reports: list[str]) -> dict:
    out = {}
    reports = [clean_report(report) for report in reports]
    for report in reports:
        # initialisation
        ipp = None
        name = None
        examen = None
        result = None
        val_ref = None
        status = None

        docs = report.split("\n")

        for doc in docs:
            # capture
            if any(p in doc for p in patient_pattern):
                if ":" in doc:
                    name = doc.split(":")[1].strip()
            if any(p in doc for p in ipp_pattern):
                rlt = re.search(r'\b[0-9]+\b', doc)
                if rlt:
                    ipp = rlt.group(0)
            if any(p in doc for p in examen_pattern):
                if ":" in doc:
                    examen = doc.split(":")[1].strip()
            if any(p in doc for p in result_pattern):
                chiffre_pattern = r'\b[0-9.]+\b'
                unite_pattern = r'\b(g/l|mg/l|g/dl)\b'
                chiffre_match = re.search(chiffre_pattern, doc)
                unite_match = re.search(unite_pattern, doc)

                if chiffre_match and unite_match:
                   result =  { 
                        "value": float(chiffre_match.group(0).strip()),
                        "unit": unite_match.group(0)
                    }
            if any(p in doc for p in val_ref_pattern):
                interval_pattern = r'\b([0-9.]+\s*-\s*[0-9.]+|[><]\s*[0-9.]+)\b'
                unite_pattern = r'\b(g/l|mg/l|d/l|g/dl)\b'
                interval_match = re.search(interval_pattern, doc)
                unite_match = re.search(unite_pattern, doc)

                if interval_match and unite_match:
                    interval = None
                    g = interval_match.group(0)
                    if "<" in g or ">" in g:
                        try:
                            val = float(g.strip().split(" ")[1])
                            interval = [0, val]
                        except:
                            interval = None
                    if "-" in g:
                        try:
                            interval = g.replace(" ", "").strip().split("-")
                            interval = [float(interval[0]), float(interval[1])]
                        except:
                            interval = None

                    if interval:
                        val_ref =  { 
                            "interval": interval,
                            "unit": unite_match.group(0)
                        }
            if any(p in doc for p in status_pattern):
                if ":" in doc:
                    status = doc.split(":")[1].strip()             
        if ipp:
            out[ipp] = {
                "ipp": ipp,
                "name": name,
                "examen": examen,
                "result": result,
                "val_ref": val_ref,
                "status": status
            }
        else:
            continue
    return out
    
build_report(reports)

{'123456': {'ipp': '123456',
  'name': 'dupont jean',
  'examen': 'hémoglobine',
  'result': {'value': 13.2, 'unit': 'g/dl'},
  'val_ref': {'interval': [12.0, 16.0], 'unit': 'g/dl'},
  'status': 'normal'},
 '789101': {'ipp': '789101',
  'name': 'martin claire',
  'examen': 'crp',
  'result': {'value': 18.0, 'unit': 'mg/l'},
  'val_ref': None,
  'status': 'anormal'},
 '456999': {'ipp': '456999',
  'name': 'durand paul',
  'examen': 'glycémie',
  'result': {'value': 1.32, 'unit': 'g/l'},
  'val_ref': {'interval': [0.7, 1.1], 'unit': 'g/l'},
  'status': None}}